# 13_dishes — v3 quality queue multihop generator

Патч для домена `dishes`: более сложные L1–L5, queue-based генерация без зависания на L2, инкрементальная запись в `dishes.jsonl`, чистые constraints и полная WDQS gold-метадата.


In [1]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which can require nbformat.
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    helper_py = _Path("common_helpers.py")
    helper_nb = _Path("00_common_helpers.ipynb")
    runner_py = _Path("notebook_runner.py")
    if helper_py.exists():
        exec(helper_py.read_text(encoding="utf-8"), globals())
    elif helper_nb.exists() and runner_py.exists():
        exec(runner_py.read_text(encoding="utf-8"), globals())
        run_ipynb(helper_nb, globals())
    else:
        raise FileNotFoundError("Need common_helpers.py or 00_common_helpers.ipynb next to this notebook")

print("✅ helpers loaded")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
✅ helpers loaded


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from __future__ import annotations

import json
import random
import re
import time
from collections import Counter
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Callable

import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm.auto import tqdm

# ============================================================
# Domain constants
# ============================================================

DISH_DOMAIN_NAME = "dishes"
Q_DISH = "Q746549"       # dish
Q_FOOD = "Q2095"         # food
Q_COUNTRY = "Q6256"      # country

# Wikidata properties used by this notebook.
P_INSTANCE = "P31"
P_SUBCLASS = "P279"
P_COUNTRY_OF_ORIGIN = "P495"
P_CUISINE = "P2012"
P_MADE_FROM = "P186"
P_HAS_PART = "P527"
P_CONTINENT = "P30"
P_OFFICIAL_LANGUAGE = "P37"
P_MEMBER_OF = "P463"
P_SHARES_BORDER_WITH = "P47"
P_COUNTRY = "P17"

DISH_GOLD_QUERY_LIMIT = 701
DISH_GOLD_LOCAL_LIMIT = 300
DISH_MIN_GOLD_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}

# 120 total: enough reserve to curate down to 100 if needed.
TARGET_PLAN_DISHES = {
    "L1": 15,
    "L2": 20,
    "L3": 25,
    "L4": 30,
    "L5": 30,
}

# Anchor countries are used only as semantic bridge constraints.
# Labels are intentionally stored here so query text / constraints do not depend on WB API label fallback.
ANCHOR_COUNTRIES = [
    {"qid": "Q142", "en": "France", "ru": "Франция"},
    {"qid": "Q38", "en": "Italy", "ru": "Италия"},
    {"qid": "Q17", "en": "Japan", "ru": "Япония"},
    {"qid": "Q668", "en": "India", "ru": "Индия"},
    {"qid": "Q148", "en": "China", "ru": "Китай"},
    {"qid": "Q29", "en": "Spain", "ru": "Испания"},
    {"qid": "Q43", "en": "Turkey", "ru": "Турция"},
    {"qid": "Q30", "en": "United States", "ru": "США"},
    {"qid": "Q183", "en": "Germany", "ru": "Германия"},
    {"qid": "Q794", "en": "Iran", "ru": "Иран"},
    {"qid": "Q159", "en": "Russia", "ru": "Россия"},
    {"qid": "Q96", "en": "Mexico", "ru": "Мексика"},
]

ANCHOR_ORGS = [
    {"qid": "Q458", "en": "European Union", "ru": "Европейский союз"},
    {"qid": "Q1065", "en": "United Nations", "ru": "Организация Объединённых Наций"},
    {"qid": "Q7159", "en": "African Union", "ru": "Африканский союз"},
    {"qid": "Q7768", "en": "ASEAN", "ru": "АСЕАН"},
    {"qid": "Q7172", "en": "Arab League", "ru": "Лига арабских государств"},
    {"qid": "Q7785", "en": "Commonwealth of Nations", "ru": "Содружество наций"},
    {"qid": "Q41550", "en": "OECD", "ru": "ОЭСР"},
]

ANCHOR_CONTINENTS = [
    {"qid": "Q46", "en": "Europe", "ru": "Европа"},
    {"qid": "Q48", "en": "Asia", "ru": "Азия"},
    {"qid": "Q49", "en": "North America", "ru": "Северная Америка"},
    {"qid": "Q18", "en": "South America", "ru": "Южная Америка"},
    {"qid": "Q15", "en": "Africa", "ru": "Африка"},
]

SEED = 1313
rng = random.Random(SEED)

print("✅ dishes constants loaded")


# ============================================================
# Reliability settings for this domain
# ============================================================
# The previous version tried to build broad pools with queries like:
#   dish -> country of origin -> country
# over all Wikidata dishes. WDQS often times out on these broad pool queries,
# and helper load_or_build_pool then returns an empty pool; generation keeps
# retrying but cannot make progress.  In this patched version pools are
# curated anchor pools by default, while every final gold list is still
# collected and validated via WDQS.
DISH_POOL_MODE = "curated"   # "curated" is robust; set to "wdqs" only for manual experiments.
DISH_OPTIONAL_WDQS_POOL_EXPANSION = False

try:
    wd.timeout = max(int(getattr(wd, "timeout", 30)), 70)
    wd.max_retries = max(int(getattr(wd, "max_retries", 4)), 6)
except Exception:
    pass

# Curated constraints. These are only anchors for making queries; all answers are
# still retrieved from Wikidata SPARQL and must satisfy ask_validator_sparql.
CURATED_CUISINES = [
    {"qid": "Q192786", "en": "Italian cuisine", "ru": "итальянская кухня"},
    {"qid": "Q234138", "en": "Japanese cuisine", "ru": "японская кухня"},
    {"qid": "Q748129", "en": "Chinese cuisine", "ru": "китайская кухня"},
    {"qid": "Q192087", "en": "Indian cuisine", "ru": "индийская кухня"},
    {"qid": "Q6661", "en": "French cuisine", "ru": "французская кухня"},
    {"qid": "Q207965", "en": "Mexican cuisine", "ru": "мексиканская кухня"},
    {"qid": "Q841984", "en": "Thai cuisine", "ru": "тайская кухня"},
    {"qid": "Q484206", "en": "Korean cuisine", "ru": "корейская кухня"},
    {"qid": "Q826059", "en": "Vietnamese cuisine", "ru": "вьетнамская кухня"},
    {"qid": "Q840090", "en": "Turkish cuisine", "ru": "турецкая кухня"},
    {"qid": "Q744027", "en": "Greek cuisine", "ru": "греческая кухня"},
    {"qid": "Q622512", "en": "Spanish cuisine", "ru": "испанская кухня"},
    {"qid": "Q1255913", "en": "Russian cuisine", "ru": "русская кухня"},
]

CURATED_INGREDIENTS = [
    {"qid": "Q5090", "en": "rice", "ru": "рис"},
    {"qid": "Q10943", "en": "cheese", "ru": "сыр"},
    {"qid": "Q23501", "en": "tomato", "ru": "томат"},
    {"qid": "Q10998", "en": "potato", "ru": "картофель"},
    {"qid": "Q93189", "en": "egg", "ru": "яйцо"},
    {"qid": "Q8495", "en": "milk", "ru": "молоко"},
    {"qid": "Q11002", "en": "sugar", "ru": "сахар"},
    {"qid": "Q11254", "en": "salt", "ru": "соль"},
    {"qid": "Q23485", "en": "onion", "ru": "лук"},
    {"qid": "Q21546392", "en": "garlic", "ru": "чеснок"},
    {"qid": "Q11575", "en": "maize", "ru": "кукуруза"},
    {"qid": "Q11006", "en": "soybean", "ru": "соя"},
    {"qid": "Q83093", "en": "mushroom", "ru": "гриб"},
    {"qid": "Q34172", "en": "butter", "ru": "сливочное масло"},
    {"qid": "Q7802", "en": "bread", "ru": "хлеб"},
    {"qid": "Q152", "en": "fish", "ru": "рыба"},
    {"qid": "Q10990", "en": "beef", "ru": "говядина"},
    {"qid": "Q10987", "en": "pork", "ru": "свинина"},
    {"qid": "Q36465", "en": "flour", "ru": "мука"},
]

CURATED_DISH_TYPES = [
    {"qid": "Q41415", "en": "soup", "ru": "суп"},
    {"qid": "Q131419", "en": "stew", "ru": "рагу"},
    {"qid": "Q9266", "en": "salad", "ru": "салат"},
    {"qid": "Q182940", "en": "dessert", "ru": "десерт"},
    {"qid": "Q28803", "en": "sandwich", "ru": "сэндвич"},
    {"qid": "Q746944", "en": "dumpling", "ru": "клёцка"},
    {"qid": "Q13276", "en": "cake", "ru": "торт"},
    {"qid": "Q44541", "en": "pancake", "ru": "блин"},
    {"qid": "Q177", "en": "pizza", "ru": "пицца"},
    {"qid": "Q46383", "en": "sushi", "ru": "суши"},
    {"qid": "Q7802", "en": "bread", "ru": "хлеб"},
]

CURATED_LANGUAGES = [
    {"qid": "Q1860", "en": "English", "ru": "английский язык"},
    {"qid": "Q150", "en": "French", "ru": "французский язык"},
    {"qid": "Q1321", "en": "Spanish", "ru": "испанский язык"},
    {"qid": "Q188", "en": "German", "ru": "немецкий язык"},
    {"qid": "Q652", "en": "Italian", "ru": "итальянский язык"},
    {"qid": "Q5287", "en": "Japanese", "ru": "японский язык"},
    {"qid": "Q7850", "en": "Chinese", "ru": "китайский язык"},
    {"qid": "Q1568", "en": "Hindi", "ru": "хинди"},
    {"qid": "Q13955", "en": "Arabic", "ru": "арабский язык"},
    {"qid": "Q7737", "en": "Russian", "ru": "русский язык"},
    {"qid": "Q256", "en": "Turkish", "ru": "турецкий язык"},
    {"qid": "Q5146", "en": "Portuguese", "ru": "португальский язык"},
]

# Seed bridge pool: dish + known origin + known ingredient + useful type anchor.
# These rows are not gold answers themselves; they only define multihop bridge constraints.
CURATED_SEED_DISHES = [
    {"dish_qid": "Q177", "dish_en": "pizza", "dish_ru": "пицца", "country_qid": "Q38", "country_en": "Italy", "country_ru": "Италия", "ingredient_qid": "Q10943", "ingredient_en": "cheese", "ingredient_ru": "сыр", "type_qid": "Q177", "type_en": "pizza", "type_ru": "пицца"},
    {"dish_qid": "Q46383", "dish_en": "sushi", "dish_ru": "суши", "country_qid": "Q17", "country_en": "Japan", "country_ru": "Япония", "ingredient_qid": "Q5090", "ingredient_en": "rice", "ingredient_ru": "рис", "type_qid": "Q46383", "type_en": "sushi", "type_ru": "суши"},
    {"dish_qid": "Q6663", "dish_en": "hamburger", "dish_ru": "гамбургер", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "ingredient_qid": "Q10990", "ingredient_en": "beef", "ingredient_ru": "говядина", "type_qid": "Q28803", "type_en": "sandwich", "type_ru": "сэндвич"},
    {"dish_qid": "Q207781", "dish_en": "croissant", "dish_ru": "круассан", "country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "ingredient_qid": "Q34172", "ingredient_en": "butter", "ingredient_ru": "сливочное масло", "type_qid": "Q207781", "type_en": "croissant", "type_ru": "круассан"},
    {"dish_qid": "Q20254", "dish_en": "lasagne", "dish_ru": "лазанья", "country_qid": "Q38", "country_en": "Italy", "country_ru": "Италия", "ingredient_qid": "Q10943", "ingredient_en": "cheese", "ingredient_ru": "сыр", "type_qid": "Q20254", "type_en": "lasagne", "type_ru": "лазанья"},
    {"dish_qid": "Q201595", "dish_en": "paella", "dish_ru": "паэлья", "country_qid": "Q29", "country_en": "Spain", "country_ru": "Испания", "ingredient_qid": "Q5090", "ingredient_en": "rice", "ingredient_ru": "рис", "type_qid": "Q201595", "type_en": "paella", "type_ru": "паэлья"},
    {"dish_qid": "Q18545", "dish_en": "falafel", "dish_ru": "фалафель", "country_qid": "Q79", "country_en": "Egypt", "country_ru": "Египет", "ingredient_qid": "Q11006", "ingredient_en": "soybean", "ingredient_ru": "соя", "type_qid": "Q18545", "type_en": "falafel", "type_ru": "фалафель"},
    {"dish_qid": "Q168155", "dish_en": "hummus", "dish_ru": "хумус", "country_qid": "Q801", "country_en": "Israel", "country_ru": "Израиль", "ingredient_qid": "Q11006", "ingredient_en": "soybean", "ingredient_ru": "соя", "type_qid": "Q168155", "type_en": "hummus", "type_ru": "хумус"},
    {"dish_qid": "Q13276", "dish_en": "cake", "dish_ru": "торт", "country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "ingredient_qid": "Q11002", "ingredient_en": "sugar", "ingredient_ru": "сахар", "type_qid": "Q13276", "type_en": "cake", "type_ru": "торт"},
    {"dish_qid": "Q44541", "dish_en": "pancake", "dish_ru": "блин", "country_qid": "Q159", "country_en": "Russia", "country_ru": "Россия", "ingredient_qid": "Q93189", "ingredient_en": "egg", "ingredient_ru": "яйцо", "type_qid": "Q44541", "type_en": "pancake", "type_ru": "блин"},
]

print("✅ dishes reliability/static pools config loaded")


✅ dishes constants loaded
✅ dishes reliability/static pools config loaded


In [3]:
# ============================================================
# Generic helpers: labels, pools, SPARQL builders, schema/meta
# ============================================================

_QID_RE = re.compile(r"^Q\d+$")


def _clean_label(x: Any) -> str:
    s = str(x or "").strip()
    return "" if _QID_RE.fullmatch(s) else s


def _ru(ent: Dict[str, Any]) -> str:
    return _clean_label(ent.get("ru")) or _clean_label(ent.get("en")) or ent.get("qid", "")


def _en(ent: Dict[str, Any]) -> str:
    return _clean_label(ent.get("en")) or _clean_label(ent.get("ru")) or ent.get("qid", "")


def _wd(qid: str) -> str:
    qid = str(qid).strip()
    if not re.fullmatch(r"Q\d+", qid):
        raise ValueError(f"Not a QID: {qid!r}")
    return f"wd:{qid}"


def _entity_from_row(row: Dict[str, Any], prefix: str) -> Dict[str, str]:
    return {
        "qid": str(row.get(f"{prefix}_qid") or row.get(prefix) or "").strip(),
        "en": _clean_label(row.get(f"{prefix}_en") or row.get(f"{prefix}LabelEn") or row.get(f"{prefix}Label")),
        "ru": _clean_label(row.get(f"{prefix}_ru") or row.get(f"{prefix}LabelRu")),
    }


def _dedupe_entities(rows: List[Dict[str, str]]) -> pd.DataFrame:
    out = []
    seen = set()
    for r in rows:
        qid = str(r.get("qid") or "").strip()
        if not re.fullmatch(r"Q\d+", qid) or qid in seen:
            continue
        en = _clean_label(r.get("en"))
        ru = _clean_label(r.get("ru"))
        if not en and not ru:
            continue
        out.append({"qid": qid, "en": en or ru, "ru": ru or en})
        seen.add(qid)
    return pd.DataFrame(out)


def _rows_to_entity_df(rows: List[Dict[str, Any]], var: str) -> pd.DataFrame:
    out = []
    for r in rows:
        qid = uri_to_qid(r.get(var, ""))
        en = _clean_label(r.get(f"{var}LabelEn") or r.get(f"{var}Label"))
        ru = _clean_label(r.get(f"{var}LabelRu"))
        if qid and (en or ru):
            out.append({"qid": qid, "en": en or ru, "ru": ru or en})
    return _dedupe_entities(out)


def _pick_df(df: pd.DataFrame, rng: random.Random) -> Dict[str, str]:
    if df is None or len(df) == 0:
        raise RuntimeError("empty pool")
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return {"qid": row["qid"], "en": row.get("en", ""), "ru": row.get("ru", "")}


def _pick_list(items: Sequence[Dict[str, str]], rng: random.Random) -> Dict[str, str]:
    if not items:
        raise RuntimeError("empty anchor list")
    return dict(rng.choice(list(items)))


def _ingredient_path(item_var: str = "item", value: str = "?ingredient") -> str:
    return f"?{item_var} (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) {value} ."


def _dish_base(item_var: str = "item") -> str:
    return f"?{item_var} wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* wd:{Q_DISH} ."


def _country_is_country(country_var: str = "country") -> str:
    return f"?{country_var} wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* wd:{Q_COUNTRY} ."


def _select_gold_sparql(where_lines: List[str], limit: int = DISH_GOLD_QUERY_LIMIT) -> str:
    where = "\n      ".join(where_lines)
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _ask_validator_sparql(where_lines: List[str]) -> str:
    where = "\n      ".join(where_lines)
    return f"""
    # WDQS-only validator. Replace {{ITEM}} with a candidate dish QID.
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      {where}
    }}
    """.strip()


def _gold_from_rows(rows: List[Dict[str, Any]], max_gold: int = DISH_GOLD_LOCAL_LIMIT) -> Tuple[List[str], List[str], List[str], Dict[str, int]]:
    seen = set()
    qids, labels_ru, labels_en = [], [], []
    ru_from_ru = 0
    ru_from_en = 0
    dropped_no_qid = 0
    dropped_no_en = 0

    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen:
            continue
        en = _clean_label(r.get("itemLabelEn"))
        ru = _clean_label(r.get("itemLabelRu"))
        if not en:
            dropped_no_en += 1
            continue
        seen.add(qid)
        qids.append(qid)
        labels_en.append(en)
        labels_ru.append(ru or en)
        if ru:
            ru_from_ru += 1
        else:
            ru_from_en += 1
        if len(qids) >= max_gold:
            break

    label_sources = {"ru_label": ru_from_ru, "en_fallback_for_ru": ru_from_en}
    stats = {"dropped_no_qid_count": dropped_no_qid, "dropped_no_en_label_count": dropped_no_en, **label_sources}
    return qids, labels_ru, labels_en, stats


def _clean_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    banned_exact = {"qid", "qids", "sparql_query", "where_lines", "created_at", "template_id", "template_family"}
    out = {}
    for k, v in (d or {}).items():
        if k in banned_exact or k.endswith("_qid") or k.endswith("_qids") or k.endswith("_ru") or k.endswith("_en"):
            continue
        if v is None or v is False or v == "" or v == [] or v == {}:
            continue
        out[k] = v
    return out


def _local_validator_meta(constraints: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": constraints,
        "label_matching_used": False,
        "note": "All constraints for this dishes task are represented directly in the WDQS ASK validator; no external local validator is required.",
    }


def _gold_meta(template_id: str, template_family: str, constraints: Dict[str, Any], rows_returned: int, gold_total: int, stats: Dict[str, int], query_limit: int, local_limit: int) -> Dict[str, Any]:
    return {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": query_limit,
        "rows_returned_by_wdqs": rows_returned,
        "gold_returned_before_limits": gold_total,
        "dropped_no_qid_count": stats.get("dropped_no_qid_count", 0),
        "dropped_no_en_label_count": stats.get("dropped_no_en_label_count", 0),
        "label_sources": {
            "ru_label": stats.get("ru_label", 0),
            "en_fallback_for_ru": stats.get("en_fallback_for_ru", 0),
        },
        "gold_may_be_incomplete_due_to_wdqs_limit": rows_returned >= query_limit,
        "quality_filter_applied": False,
        "constraints_are_wdqs_only": True,
        "constraints": constraints,
        "gold_limit": local_limit,
        "gold_returned": gold_total,
        "gold_total_before_limit": gold_total,
        "gold_truncated_by_local_limit": gold_total >= local_limit,
        "template_id": template_id,
        "template_family": template_family,
    }


def _example_to_dict(ex):
    if is_dataclass(ex):
        return asdict(ex)
    if hasattr(ex, "__dict__"):
        return dict(ex.__dict__)
    return dict(ex)

print("✅ generic dishes helpers loaded")


✅ generic dishes helpers loaded


In [4]:
# ============================================================
# Robust Wikidata pools
# ============================================================
# Default mode is curated anchor pools. This fixes the WDQS timeout shown when
# building 13_dishes_country_pool_v3: the broad pool query was too expensive.
# Gold answers are still collected only through SPARQL in _build_dish_record().


def _curated_entity_df(items: Sequence[Dict[str, str]]) -> pd.DataFrame:
    return _dedupe_entities([
        {"qid": str(x.get("qid", "")).strip(), "en": _clean_label(x.get("en")), "ru": _clean_label(x.get("ru"))}
        for x in items
    ])


def _curated_seed_df(items: Sequence[Dict[str, str]]) -> pd.DataFrame:
    rows = []
    for r in items:
        row = {k: str(v or "").strip() for k, v in r.items()}
        required_qids = ["dish_qid", "country_qid", "ingredient_qid", "type_qid"]
        if not all(re.fullmatch(r"Q\d+", row.get(k, "")) for k in required_qids):
            continue
        if not row.get("dish_en") or not row.get("ingredient_en") or not row.get("type_en"):
            continue
        rows.append(row)
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


# ----- Optional old WDQS builders, kept only for manual experiments -----

def _build_country_pool_wdqs() -> pd.DataFrame:
    # More bounded than the old query: VALUES over anchor countries, not all countries in Wikidata.
    values = " ".join(_wd(c["qid"]) for c in ANCHOR_COUNTRIES)
    sparql = f"""
    SELECT DISTINCT ?country ?countryLabelEn ?countryLabelRu WHERE {{
      VALUES ?country {{ {values} }}
      ?country rdfs:label ?countryLabelEn FILTER(LANG(?countryLabelEn) = "en") .
      OPTIONAL {{ ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") . }}
    }}
    """
    df = _rows_to_entity_df(rows_from_select(wd.sparql_select(sparql)), "country")
    return df if len(df) else _curated_entity_df(ANCHOR_COUNTRIES)


def _build_cuisine_pool_wdqs() -> pd.DataFrame:
    values = " ".join(_wd(c["qid"]) for c in CURATED_CUISINES)
    sparql = f"""
    SELECT DISTINCT ?cuisine ?cuisineLabelEn ?cuisineLabelRu WHERE {{
      VALUES ?cuisine {{ {values} }}
      ?cuisine rdfs:label ?cuisineLabelEn FILTER(LANG(?cuisineLabelEn) = "en") .
      OPTIONAL {{ ?cuisine rdfs:label ?cuisineLabelRu FILTER(LANG(?cuisineLabelRu) = "ru") . }}
    }}
    """
    df = _rows_to_entity_df(rows_from_select(wd.sparql_select(sparql)), "cuisine")
    return df if len(df) else _curated_entity_df(CURATED_CUISINES)


def _build_ingredient_pool_wdqs() -> pd.DataFrame:
    values = " ".join(_wd(c["qid"]) for c in CURATED_INGREDIENTS)
    sparql = f"""
    SELECT DISTINCT ?ingredient ?ingredientLabelEn ?ingredientLabelRu WHERE {{
      VALUES ?ingredient {{ {values} }}
      ?ingredient rdfs:label ?ingredientLabelEn FILTER(LANG(?ingredientLabelEn) = "en") .
      OPTIONAL {{ ?ingredient rdfs:label ?ingredientLabelRu FILTER(LANG(?ingredientLabelRu) = "ru") . }}
    }}
    """
    df = _rows_to_entity_df(rows_from_select(wd.sparql_select(sparql)), "ingredient")
    return df if len(df) else _curated_entity_df(CURATED_INGREDIENTS)


def _build_dish_type_pool_wdqs() -> pd.DataFrame:
    values = " ".join(_wd(c["qid"]) for c in CURATED_DISH_TYPES)
    sparql = f"""
    SELECT DISTINCT ?type ?typeLabelEn ?typeLabelRu WHERE {{
      VALUES ?type {{ {values} }}
      ?type rdfs:label ?typeLabelEn FILTER(LANG(?typeLabelEn) = "en") .
      OPTIONAL {{ ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") . }}
    }}
    """
    df = _rows_to_entity_df(rows_from_select(wd.sparql_select(sparql)), "type")
    return df if len(df) else _curated_entity_df(CURATED_DISH_TYPES)


def _build_language_pool_wdqs() -> pd.DataFrame:
    values = " ".join(_wd(c["qid"]) for c in CURATED_LANGUAGES)
    sparql = f"""
    SELECT DISTINCT ?language ?languageLabelEn ?languageLabelRu WHERE {{
      VALUES ?language {{ {values} }}
      ?language rdfs:label ?languageLabelEn FILTER(LANG(?languageLabelEn) = "en") .
      OPTIONAL {{ ?language rdfs:label ?languageLabelRu FILTER(LANG(?languageLabelRu) = "ru") . }}
    }}
    """
    df = _rows_to_entity_df(rows_from_select(wd.sparql_select(sparql)), "language")
    return df if len(df) else _curated_entity_df(CURATED_LANGUAGES)


def _build_seed_dish_pool_wdqs() -> pd.DataFrame:
    # Bounded verification/enrichment over curated seeds only.
    values = " ".join(_wd(c["dish_qid"]) for c in CURATED_SEED_DISHES)
    sparql = f"""
    SELECT DISTINCT ?dish ?dishLabelEn ?dishLabelRu ?country ?countryLabelEn ?countryLabelRu ?ingredient ?ingredientLabelEn ?ingredientLabelRu ?type ?typeLabelEn ?typeLabelRu WHERE {{
      VALUES ?dish {{ {values} }}
      ?dish wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* wd:{Q_DISH} .
      OPTIONAL {{ ?dish wdt:{P_COUNTRY_OF_ORIGIN} ?country . }}
      OPTIONAL {{ ?dish (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?ingredient . }}
      OPTIONAL {{ ?dish wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* ?type . ?type wdt:{P_SUBCLASS}* wd:{Q_DISH} . FILTER(?type != wd:{Q_DISH}) }}
      ?dish rdfs:label ?dishLabelEn FILTER(LANG(?dishLabelEn) = "en") .
      OPTIONAL {{ ?dish rdfs:label ?dishLabelRu FILTER(LANG(?dishLabelRu) = "ru") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelEn FILTER(LANG(?countryLabelEn) = "en") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") . }}
      OPTIONAL {{ ?ingredient rdfs:label ?ingredientLabelEn FILTER(LANG(?ingredientLabelEn) = "en") . }}
      OPTIONAL {{ ?ingredient rdfs:label ?ingredientLabelRu FILTER(LANG(?ingredientLabelRu) = "ru") . }}
      OPTIONAL {{ ?type rdfs:label ?typeLabelEn FILTER(LANG(?typeLabelEn) = "en") . }}
      OPTIONAL {{ ?type rdfs:label ?typeLabelRu FILTER(LANG(?typeLabelRu) = "ru") . }}
    }}
    LIMIT 500
    """
    # This experimental branch is intentionally not the default; fallback is enough.
    try:
        rows = rows_from_select(wd.sparql_select(sparql))
    except Exception:
        return _curated_seed_df(CURATED_SEED_DISHES)
    out = []
    for r in rows:
        dish = _entity_from_row(r, "dish")
        country = _entity_from_row(r, "country")
        ingredient = _entity_from_row(r, "ingredient")
        typ = _entity_from_row(r, "type")
        if not re.fullmatch(r"Q\d+", dish.get("qid", "")):
            continue
        # If optional details are missing in Wikidata, retain curated bridge details below.
        cur = next((x for x in CURATED_SEED_DISHES if x["dish_qid"] == dish["qid"]), None)
        if cur:
            out.append(cur)
    df = _curated_seed_df(out)
    return df if len(df) else _curated_seed_df(CURATED_SEED_DISHES)


# ----- Builders used by load_or_build_pool -----

def _build_country_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_country_pool_wdqs()
    return _curated_entity_df(ANCHOR_COUNTRIES)


def _build_cuisine_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_cuisine_pool_wdqs()
    return _curated_entity_df(CURATED_CUISINES)


def _build_ingredient_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_ingredient_pool_wdqs()
    return _curated_entity_df(CURATED_INGREDIENTS)


def _build_dish_type_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_dish_type_pool_wdqs()
    return _curated_entity_df(CURATED_DISH_TYPES)


def _build_seed_dish_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_seed_dish_pool_wdqs()
    return _curated_seed_df(CURATED_SEED_DISHES)


def _build_language_pool() -> pd.DataFrame:
    if DISH_POOL_MODE == "wdqs" and DISH_OPTIONAL_WDQS_POOL_EXPANSION:
        return _build_language_pool_wdqs()
    return _curated_entity_df(CURATED_LANGUAGES)


# New pool cache names v4 avoid reusing failed/heavy v3 pool files.
def country_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_country_pool_v4_curated", _build_country_pool)


def cuisine_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_cuisine_pool_v4_curated", _build_cuisine_pool)


def ingredient_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_ingredient_pool_v4_curated", _build_ingredient_pool)


def dish_type_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_type_pool_v4_curated", _build_dish_type_pool)


def seed_dish_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_seed_pool_v4_curated", _build_seed_dish_pool)


def language_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_language_pool_v4_curated", _build_language_pool)


def _pick_seed(rng: random.Random) -> Dict[str, str]:
    df = seed_dish_pool()
    if df is None or len(df) == 0:
        raise RuntimeError("empty seed dish pool")
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0].to_dict()
    return {k: str(v) for k, v in row.items()}

print("✅ dishes pools defined: curated anchors by default, no broad WDQS pool build")


✅ dishes pools defined: curated anchors by default, no broad WDQS pool build


In [5]:

# ============================================================
# v3 quality patch: richer constraints, no simple one-criterion L1
# ============================================================

DISHES_PATCH_VERSION = "v3_quality_queue"

# Для этого домена держим requested_count=5 везде, чтобы формат был ближе к остальным curated-доменам.
DISH_REQUESTED_COUNT_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 5, "L4": 5, "L5": 5}
DISH_MIN_GOLD_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 5, "L4": 5, "L5": 5}

# 120 записей с запасом для дальнейшей ручной чистки.
TARGET_PLAN_DISHES = {"L1": 15, "L2": 20, "L3": 25, "L4": 30, "L5": 30}

# Больше лимиты для сложных уровней, но генерация больше не висит: каждый кандидат пробуется один раз.
DISH_MAX_CANDIDATES_PER_LEVEL = {"L1": 900, "L2": 1800, "L3": 2400, "L4": 3200, "L5": 4200}
DISH_MAX_TEMPLATE_CANDIDATES = {"L1": 250, "L2": 350, "L3": 450, "L4": 550, "L5": 700}

# Soft gold-overlap dedup внутри текущего output, чтобы не принимать почти одинаковые answer sets.
DISH_DEDUP_GOLD_JACCARD_THRESHOLD = 0.92
DISH_DEDUP_GOLD_CONTAINMENT_THRESHOLD = 0.98
DISH_DEDUP_GOLD_SIZE_RATIO_THRESHOLD = 0.75

# Current membership with statement qualifier support.
P_END_TIME = "P582"


def _extend_unique_entities(base: Sequence[Dict[str, str]], extra: Sequence[Dict[str, str]]) -> List[Dict[str, str]]:
    out, seen = [], set()
    for x in list(base or []) + list(extra or []):
        q = str(x.get("qid", "")).strip()
        if not re.fullmatch(r"Q\d+", q) or q in seen:
            continue
        out.append({"qid": q, "en": _clean_label(x.get("en")), "ru": _clean_label(x.get("ru"))})
        seen.add(q)
    return out


# Расширяем anchor-пулы, чтобы не было перекоса в несколько кухонь / стран.
ANCHOR_COUNTRIES = _extend_unique_entities(ANCHOR_COUNTRIES, [
    {"qid": "Q79", "en": "Egypt", "ru": "Египет"},
    {"qid": "Q801", "en": "Israel", "ru": "Израиль"},
    {"qid": "Q155", "en": "Brazil", "ru": "Бразилия"},
    {"qid": "Q414", "en": "Argentina", "ru": "Аргентина"},
    {"qid": "Q884", "en": "South Korea", "ru": "Республика Корея"},
    {"qid": "Q869", "en": "Thailand", "ru": "Таиланд"},
    {"qid": "Q881", "en": "Vietnam", "ru": "Вьетнам"},
    {"qid": "Q252", "en": "Indonesia", "ru": "Индонезия"},
    {"qid": "Q843", "en": "Pakistan", "ru": "Пакистан"},
    {"qid": "Q212", "en": "Ukraine", "ru": "Украина"},
    {"qid": "Q34", "en": "Sweden", "ru": "Швеция"},
    {"qid": "Q36", "en": "Poland", "ru": "Польша"},
    {"qid": "Q1033", "en": "Nigeria", "ru": "Нигерия"},
    {"qid": "Q1028", "en": "Morocco", "ru": "Марокко"},
    {"qid": "Q822", "en": "Lebanon", "ru": "Ливан"},
    {"qid": "Q39", "en": "Switzerland", "ru": "Швейцария"},
    {"qid": "Q28", "en": "Hungary", "ru": "Венгрия"},
    {"qid": "Q55", "en": "Netherlands", "ru": "Нидерланды"},
])

ANCHOR_CONTINENTS = _extend_unique_entities(ANCHOR_CONTINENTS, [
    {"qid": "Q55643", "en": "Oceania", "ru": "Океания"},
])

ANCHOR_ORGS = _extend_unique_entities(ANCHOR_ORGS, [
    {"qid": "Q7795", "en": "OPEC", "ru": "ОПЕК"},
    {"qid": "Q4264", "en": "Mercosur", "ru": "Меркосур"},
])

# Используем новые имена кэшей, чтобы не подхватывать старые пустые/сломанные v2-v4 cache-файлы.
def country_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_country_pool_v5_quality", lambda: _curated_entity_df(ANCHOR_COUNTRIES))


def cuisine_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_cuisine_pool_v5_quality", lambda: _curated_entity_df(CURATED_CUISINES))


def ingredient_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_ingredient_pool_v5_quality", lambda: _curated_entity_df(CURATED_INGREDIENTS))


def dish_type_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_type_pool_v5_quality", lambda: _curated_entity_df(CURATED_DISH_TYPES))


def seed_dish_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_seed_pool_v5_quality", lambda: _curated_seed_df(CURATED_SEED_DISHES))


def language_pool() -> pd.DataFrame:
    return load_or_build_pool("13_dishes_language_pool_v5_quality", lambda: _curated_entity_df(CURATED_LANGUAGES))


def _all_df(df: pd.DataFrame) -> List[Dict[str, str]]:
    if df is None or len(df) == 0:
        return []
    return [{"qid": str(r["qid"]), "en": str(r.get("en", "")), "ru": str(r.get("ru", ""))} for r in df.to_dict("records")]


def _requested_count(complexity: str, rng: Optional[random.Random] = None) -> int:
    return int(DISH_REQUESTED_COUNT_BY_LEVEL.get(complexity, 5))


def _country_current_member_lines(country_var: str, org_qid: str, suffix: str = "") -> List[str]:
    # Current-ish membership: statement P463 with no P582 end time. Much safer than bare wdt:P463 for org constraints.
    st = f"?{country_var}MembershipStatement{suffix}"
    end = f"?{country_var}MembershipEnd{suffix}"
    return [
        f"?{country_var} p:{P_MEMBER_OF} {st} .",
        f"{st} ps:{P_MEMBER_OF} {_wd(org_qid)} .",
        f"FILTER NOT EXISTS {{ {st} pq:{P_END_TIME} {end} . }}",
    ]



def _period(s: str) -> str:
    s = str(s or "").strip()
    return s if s.endswith(".") else s + "."


def _spec(template_id: str, family: str, where_lines: List[str], constraints: Dict[str, Any], ru: str, en: str, requested_count: int) -> Dict[str, Any]:
    return {
        "template_id": template_id,
        "template_family": family,
        "where_lines": where_lines,
        "constraints": _clean_constraints(constraints),
        "query_text_ru": _period(ru),
        "query_text_en": _period(en),
        "requested_count": int(requested_count),
    }


def _head_ru(n: int) -> str:
    return f"Назови {n} блюд"


def _head_en(n: int) -> str:
    return f"Name {n} dishes"


def _dish_type_line(typ: Dict[str, str], item_var: str = "item") -> str:
    return f"?{item_var} wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* {_wd(typ['qid'])} ."


def _origin_country_lines(country_var: str = "country") -> List[str]:
    return [f"?item wdt:{P_COUNTRY_OF_ORIGIN} ?{country_var} .", _country_is_country(country_var)]


# -------------------------
# L1: теперь минимум два критерия
# -------------------------

def _make_l1_country_ingredient(c: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), f"?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(c['qid'])} .", _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l1_country_ingredient", "origin_country_ingredient", where,
        {"kind": "dish", "country_of_origin": _en(c), "ingredient": _en(ing)},
        f"{_head_ru(n)}, происходящих из страны «{_ru(c)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} whose country of origin is {_en(c)} and that contain {_en(ing)}",
        n,
    )


def _make_l1_country_type(c: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), f"?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(c['qid'])} .", _dish_type_line(typ)]
    return _spec(
        "dishes_l1_country_type", "origin_country_type", where,
        {"kind": "dish", "country_of_origin": _en(c), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих из страны «{_ru(c)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin is {_en(c)}",
        n,
    )


def _make_l1_cuisine_ingredient(cuis: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l1_cuisine_ingredient", "cuisine_ingredient", where,
        {"kind": "dish", "cuisine": _en(cuis), "ingredient": _en(ing)},
        f"{_head_ru(n)} кухни «{_ru(cuis)}», содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} from {_en(cuis)} cuisine that contain {_en(ing)}",
        n,
    )


def _make_l1_cuisine_type(cuis: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", _dish_type_line(typ)]
    return _spec(
        "dishes_l1_cuisine_type", "cuisine_type", where,
        {"kind": "dish", "cuisine": _en(cuis), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», относящихся к кухне «{_ru(cuis)}»",
        f"{_head_en(n)} of dish type {_en(typ)} that belong to {_en(cuis)} cuisine",
        n,
    )


def _make_l1_ingredient_type(ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l1_ingredient_type", "ingredient_type", where,
        {"kind": "dish", "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} that contain {_en(ing)}",
        n,
    )


def _make_l1_continent_ingredient(cont: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L1")
    where = [_dish_base(), *_origin_country_lines("country"), f"?country wdt:{P_CONTINENT} {_wd(cont['qid'])} .", _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l1_continent_ingredient", "continent_ingredient", where,
        {"kind": "dish", "country_continent": _en(cont), "ingredient": _en(ing)},
        f"{_head_ru(n)}, происходящих из стран континента «{_ru(cont)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} whose country of origin is in {_en(cont)} and that contain {_en(ing)}",
        n,
    )


# -------------------------
# L2: три критерия, но high-recall
# -------------------------

def _make_l2_country_ingredient_type(c: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [_dish_base(), f"?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(c['qid'])} .", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l2_country_ingredient_type", "origin_country_ingredient_type", where,
        {"kind": "dish", "country_of_origin": _en(c), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих из страны «{_ru(c)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin is {_en(c)} and that contain {_en(ing)}",
        n,
    )


def _make_l2_cuisine_ingredient_type(cuis: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l2_cuisine_ingredient_type", "cuisine_ingredient_type", where,
        {"kind": "dish", "cuisine": _en(cuis), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}» из кухни «{_ru(cuis)}», содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} from {_en(cuis)} cuisine that contain {_en(ing)}",
        n,
    )


def _make_l2_continent_ingredient_type(cont: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [_dish_base(), *_origin_country_lines("country"), f"?country wdt:{P_CONTINENT} {_wd(cont['qid'])} .", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l2_continent_ingredient_type", "continent_ingredient_type", where,
        {"kind": "dish", "country_continent": _en(cont), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих из стран континента «{_ru(cont)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin is in {_en(cont)} and that contain {_en(ing)}",
        n,
    )


def _make_l2_org_ingredient_type(org: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [_dish_base(), *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org"), _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l2_org_ingredient_type", "country_org_ingredient_type", where,
        {"kind": "dish", "country_member_of": _en(org), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих из стран — текущих членов «{_ru(org)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin is a current member of {_en(org)} and that contain {_en(ing)}",
        n,
    )


def _make_l2_language_ingredient_type(lang: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [_dish_base(), *_origin_country_lines("country"), f"?country wdt:{P_OFFICIAL_LANGUAGE} {_wd(lang['qid'])} .", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l2_language_ingredient_type", "country_language_ingredient_type", where,
        {"kind": "dish", "country_official_language": _en(lang), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих из стран с официальным языком «{_ru(lang)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin has {_en(lang)} as an official language and that contain {_en(ing)}",
        n,
    )

print("✅ dishes v3 L1/L2 quality templates loaded")


✅ dishes v3 L1/L2 quality templates loaded


In [6]:

# ============================================================
# v3 L3-L5 richer multihop templates
# ============================================================


def _make_l3_continent_cuisine_ingredient(cont: Dict[str, str], cuis: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L3")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", *_origin_country_lines("country"), f"?country wdt:{P_CONTINENT} {_wd(cont['qid'])} .", _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l3_continent_cuisine_ingredient", "continent_cuisine_ingredient", where,
        {"kind": "dish", "country_continent": _en(cont), "cuisine": _en(cuis), "ingredient": _en(ing)},
        f"{_head_ru(n)} кухни «{_ru(cuis)}», происходящих из стран континента «{_ru(cont)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} from {_en(cuis)} cuisine whose country of origin is in {_en(cont)} and that contain {_en(ing)}",
        n,
    )


def _make_l3_org_cuisine_ingredient(org: Dict[str, str], cuis: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L3")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org"), _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l3_org_cuisine_ingredient", "country_org_cuisine_ingredient", where,
        {"kind": "dish", "country_member_of": _en(org), "cuisine": _en(cuis), "ingredient": _en(ing)},
        f"{_head_ru(n)} кухни «{_ru(cuis)}», происходящих из стран — текущих членов «{_ru(org)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} from {_en(cuis)} cuisine whose country of origin is a current member of {_en(org)} and that contain {_en(ing)}",
        n,
    )


def _make_l3_border_ingredient_type(anchor: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L3")
    where = [_dish_base(), *_origin_country_lines("country"), f"?country wdt:{P_SHARES_BORDER_WITH} {_wd(anchor['qid'])} .", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l3_border_ingredient_type", "border_country_ingredient_type", where,
        {"kind": "dish", "origin_country_borders": _en(anchor), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», страна происхождения которых граничит с «{_ru(anchor)}», и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin borders {_en(anchor)} and that contain {_en(ing)}",
        n,
    )


def _make_l3_seed_shared_ingredient_type(seed: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L3")
    seed_ent = {"qid": seed["dish_qid"], "en": seed["dish_en"], "ru": seed["dish_ru"]}
    where = [_dish_base(), f"{_wd(seed['dish_qid'])} (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?bridgeIngredient .", _ingredient_path("item", "?bridgeIngredient"), _dish_type_line(typ), f"FILTER(?item != {_wd(seed['dish_qid'])})"]
    return _spec(
        "dishes_l3_shared_ingredient_with_seed_type", "seed_shared_ingredient_type", where,
        {"kind": "dish", "shares_ingredient_with_dish": _en(seed_ent), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», у которых есть хотя бы один общий ингредиент с блюдом «{_ru(seed_ent)}»",
        f"{_head_en(n)} of dish type {_en(typ)} that share at least one ingredient with the dish \"{_en(seed_ent)}\"",
        n,
    )


def _make_l3_language_cuisine_type(lang: Dict[str, str], cuis: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L3")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", *_origin_country_lines("country"), f"?country wdt:{P_OFFICIAL_LANGUAGE} {_wd(lang['qid'])} .", _dish_type_line(typ)]
    return _spec(
        "dishes_l3_language_cuisine_type", "country_language_cuisine_type", where,
        {"kind": "dish", "country_official_language": _en(lang), "cuisine": _en(cuis), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}» из кухни «{_ru(cuis)}», происходящих из стран с официальным языком «{_ru(lang)}»",
        f"{_head_en(n)} of dish type {_en(typ)} from {_en(cuis)} cuisine whose country of origin has {_en(lang)} as an official language",
        n,
    )


# -------------------------
# L4
# -------------------------

def _make_l4_same_continent_ingredient_type(anchor: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L4")
    where = [_dish_base(), f"{_wd(anchor['qid'])} wdt:{P_CONTINENT} ?bridgeContinent .", *_origin_country_lines("country"), f"?country wdt:{P_CONTINENT} ?bridgeContinent .", f"FILTER(?country != {_wd(anchor['qid'])})", f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(anchor['qid'])} . }}", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l4_same_continent_ingredient_type", "same_continent_bridge_ingredient_type", where,
        {"kind": "dish", "origin_country_same_continent_as": _en(anchor), "exclude_country_of_origin": _en(anchor), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих не из «{_ru(anchor)}», а из других стран на том же континенте, и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} not from {_en(anchor)} but from other countries on the same continent, and containing {_en(ing)}",
        n,
    )


def _make_l4_shared_language_cuisine_ingredient(anchor: Dict[str, str], cuis: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L4")
    where = [_dish_base(), f"{_wd(anchor['qid'])} wdt:{P_OFFICIAL_LANGUAGE} ?bridgeLanguage .", *_origin_country_lines("country"), f"?country wdt:{P_OFFICIAL_LANGUAGE} ?bridgeLanguage .", f"FILTER(?country != {_wd(anchor['qid'])})", f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(anchor['qid'])} . }}", f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .", _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l4_shared_language_cuisine_ingredient", "shared_language_cuisine_ingredient", where,
        {"kind": "dish", "origin_country_shares_official_language_with": _en(anchor), "exclude_country_of_origin": _en(anchor), "cuisine": _en(cuis), "ingredient": _en(ing)},
        f"{_head_ru(n)} кухни «{_ru(cuis)}», происходящих не из «{_ru(anchor)}», но из стран с общим с ней официальным языком, и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} from {_en(cuis)} cuisine, not from {_en(anchor)} but from countries sharing an official language with it, and containing {_en(ing)}",
        n,
    )


def _make_l4_source_country_ingredient_org_type(source_country: Dict[str, str], org: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L4")
    where = [_dish_base(), f"?sourceDish wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* wd:{Q_DISH} .", f"?sourceDish wdt:{P_COUNTRY_OF_ORIGIN} {_wd(source_country['qid'])} .", f"?sourceDish (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?bridgeIngredient .", _ingredient_path("item", "?bridgeIngredient"), *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org"), f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(source_country['qid'])} . }}", _dish_type_line(typ)]
    return _spec(
        "dishes_l4_ingredient_from_source_country_org_type", "source_country_ingredient_org_type", where,
        {"kind": "dish", "has_ingredient_used_in_dishes_from_country": _en(source_country), "exclude_country_of_origin": _en(source_country), "country_member_of": _en(org), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих не из «{_ru(source_country)}», но содержащих ингредиент из блюд этой страны; страна происхождения блюда должна быть текущим членом «{_ru(org)}»",
        f"{_head_en(n)} of dish type {_en(typ)} not from {_en(source_country)} but containing an ingredient used in dishes from that country; the dish country of origin must be a current member of {_en(org)}",
        n,
    )


def _make_l4_seed_type_org_ingredient(seed: Dict[str, str], org: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L4")
    seed_ent = {"qid": seed["dish_qid"], "en": seed["dish_en"], "ru": seed["dish_ru"]}
    where = [_dish_base(), f"{_wd(seed['dish_qid'])} wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* ?bridgeType .", f"?bridgeType wdt:{P_SUBCLASS}* wd:{Q_DISH} .", f"?item wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* ?bridgeType .", f"FILTER(?item != {_wd(seed['dish_qid'])})", *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org"), _ingredient_path("item", _wd(ing["qid"]))]
    return _spec(
        "dishes_l4_same_type_as_seed_org_ingredient", "seed_type_org_ingredient", where,
        {"kind": "dish", "same_dish_type_as": _en(seed_ent), "country_member_of": _en(org), "ingredient": _en(ing)},
        f"{_head_ru(n)} того же типа, что и «{_ru(seed_ent)}», происходящих из стран — текущих членов «{_ru(org)}» и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of the same dish type as \"{_en(seed_ent)}\", whose country of origin is a current member of {_en(org)} and that contain {_en(ing)}",
        n,
    )


# -------------------------
# L5
# -------------------------

def _make_l5_same_continent_org_ingredient_type(anchor: Dict[str, str], org: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L5")
    where = [_dish_base(), f"{_wd(anchor['qid'])} wdt:{P_CONTINENT} ?bridgeContinent .", *_origin_country_lines("country"), f"?country wdt:{P_CONTINENT} ?bridgeContinent .", f"FILTER(?country != {_wd(anchor['qid'])})", f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(anchor['qid'])} . }}", *_country_current_member_lines("country", org["qid"], "Org"), _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l5_same_continent_org_ingredient_type", "same_continent_org_ingredient_type", where,
        {"kind": "dish", "origin_country_same_continent_as": _en(anchor), "exclude_country_of_origin": _en(anchor), "country_member_of": _en(org), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих не из «{_ru(anchor)}», а из стран на том же континенте, которые являются текущими членами «{_ru(org)}», и содержащих ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} not from {_en(anchor)} but from countries on the same continent that are current members of {_en(org)}, and containing {_en(ing)}",
        n,
    )


def _make_l5_shared_language_seed_ingredient_type(anchor: Dict[str, str], seed: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L5")
    seed_ent = {"qid": seed["dish_qid"], "en": seed["dish_en"], "ru": seed["dish_ru"]}
    where = [_dish_base(), f"{_wd(anchor['qid'])} wdt:{P_OFFICIAL_LANGUAGE} ?bridgeLanguage .", *_origin_country_lines("country"), f"?country wdt:{P_OFFICIAL_LANGUAGE} ?bridgeLanguage .", f"FILTER(?country != {_wd(anchor['qid'])})", f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(anchor['qid'])} . }}", f"{_wd(seed['dish_qid'])} (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?bridgeIngredient .", _ingredient_path("item", "?bridgeIngredient"), _dish_type_line(typ), f"FILTER(?item != {_wd(seed['dish_qid'])})"]
    return _spec(
        "dishes_l5_shared_language_seed_ingredient_type", "shared_language_seed_ingredient_type", where,
        {"kind": "dish", "origin_country_shares_official_language_with": _en(anchor), "exclude_country_of_origin": _en(anchor), "shares_ingredient_with_dish": _en(seed_ent), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», происходящих не из «{_ru(anchor)}», но из стран с общим с ней официальным языком, и имеющих общий ингредиент с блюдом «{_ru(seed_ent)}»",
        f"{_head_en(n)} of dish type {_en(typ)} not from {_en(anchor)} but from countries sharing an official language with it, and sharing an ingredient with the dish \"{_en(seed_ent)}\"",
        n,
    )


def _make_l5_border_source_ingredient_type(border_anchor: Dict[str, str], source_country: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L5")
    where = [_dish_base(), f"?sourceDish wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* wd:{Q_DISH} .", f"?sourceDish wdt:{P_COUNTRY_OF_ORIGIN} {_wd(source_country['qid'])} .", f"?sourceDish (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?bridgeIngredient .", _ingredient_path("item", "?bridgeIngredient"), *_origin_country_lines("country"), f"?country wdt:{P_SHARES_BORDER_WITH} {_wd(border_anchor['qid'])} .", f"FILTER(?country != {_wd(source_country['qid'])})", f"FILTER NOT EXISTS {{ ?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(source_country['qid'])} . }}", _dish_type_line(typ)]
    return _spec(
        "dishes_l5_border_source_ingredient_type", "border_country_source_ingredient_type", where,
        {"kind": "dish", "origin_country_borders": _en(border_anchor), "has_ingredient_used_in_dishes_from_country": _en(source_country), "exclude_country_of_origin": _en(source_country), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», страна происхождения которых граничит с «{_ru(border_anchor)}»; блюда должны содержать ингредиент, встречающийся в блюдах из «{_ru(source_country)}», но сами происходить не из этой страны",
        f"{_head_en(n)} of dish type {_en(typ)} whose country of origin borders {_en(border_anchor)}; the dishes must contain an ingredient used in dishes from {_en(source_country)}, while not originating from that country",
        n,
    )


def _make_l5_cuisine_country_match_org_ingredient_type(org: Dict[str, str], ing: Dict[str, str], typ: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L5")
    where = [_dish_base(), f"?item wdt:{P_CUISINE} ?cuisine .", *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org"), f"{{ ?cuisine wdt:{P_COUNTRY_OF_ORIGIN} ?country . }} UNION {{ ?cuisine wdt:{P_COUNTRY} ?country . }}", _ingredient_path("item", _wd(ing["qid"])), _dish_type_line(typ)]
    return _spec(
        "dishes_l5_cuisine_country_match_org_ingredient_type", "cuisine_country_match_org_ingredient_type", where,
        {"kind": "dish", "cuisine_country_matches_origin_country": True, "country_member_of": _en(org), "ingredient": _en(ing), "dish_type": _en(typ)},
        f"{_head_ru(n)} типа «{_ru(typ)}», у которых кухня связана с той же страной, что и страна происхождения; эта страна должна быть текущим членом «{_ru(org)}», а в составе должен быть ингредиент «{_ru(ing)}»",
        f"{_head_en(n)} of dish type {_en(typ)} whose cuisine is linked to the same country as the dish country of origin; that country must be a current member of {_en(org)}, and the dish must contain {_en(ing)}",
        n,
    )


def _make_l5_seed_type_org_shared_ingredient(seed: Dict[str, str], org: Dict[str, str], ing: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L5")
    seed_ent = {"qid": seed["dish_qid"], "en": seed["dish_en"], "ru": seed["dish_ru"]}
    where = [_dish_base(), f"{_wd(seed['dish_qid'])} wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* ?bridgeType .", f"?bridgeType wdt:{P_SUBCLASS}* wd:{Q_DISH} .", f"?item wdt:{P_INSTANCE}/wdt:{P_SUBCLASS}* ?bridgeType .", f"FILTER(?item != {_wd(seed['dish_qid'])})", f"{_wd(seed['dish_qid'])} (wdt:{P_HAS_PART}|wdt:{P_MADE_FROM}) ?seedIngredient .", _ingredient_path("item", "?seedIngredient"), _ingredient_path("item", _wd(ing["qid"])), *_origin_country_lines("country"), *_country_current_member_lines("country", org["qid"], "Org")]
    return _spec(
        "dishes_l5_seed_type_org_shared_and_fixed_ingredient", "seed_type_org_two_ingredient_bridge", where,
        {"kind": "dish", "same_dish_type_as": _en(seed_ent), "shares_ingredient_with_dish": _en(seed_ent), "country_member_of": _en(org), "ingredient": _en(ing)},
        f"{_head_ru(n)} того же типа, что и «{_ru(seed_ent)}», с общим с ним ингредиентом; страна происхождения должна быть текущим членом «{_ru(org)}», а блюдо также должно содержать «{_ru(ing)}»",
        f"{_head_en(n)} of the same dish type as \"{_en(seed_ent)}\" and sharing an ingredient with it; the country of origin must be a current member of {_en(org)}, and the dish must also contain {_en(ing)}",
        n,
    )

print("✅ dishes v3 L3-L5 multihop templates loaded")


✅ dishes v3 L3-L5 multihop templates loaded


In [7]:

# ============================================================
# v3 queue-based record builder / generator
# ============================================================

DISH_TEMPLATE_FAMILIES_BY_LEVEL = {
    "L1": [
        "country_ingredient", "country_type", "cuisine_ingredient", "cuisine_type", "ingredient_type", "continent_ingredient",
    ],
    "L2": [
        "country_ingredient_type", "cuisine_ingredient_type", "continent_ingredient_type", "org_ingredient_type", "language_ingredient_type",
    ],
    "L3": [
        "continent_cuisine_ingredient", "org_cuisine_ingredient", "border_ingredient_type", "seed_shared_ingredient_type", "language_cuisine_type",
    ],
    "L4": [
        "same_continent_ingredient_type", "shared_language_cuisine_ingredient", "source_country_ingredient_org_type", "seed_type_org_ingredient",
    ],
    "L5": [
        "same_continent_org_ingredient_type", "shared_language_seed_ingredient_type", "border_source_ingredient_type", "cuisine_country_match_org_ingredient_type", "seed_type_org_shared_ingredient",
    ],
}


def _candidate_signature(spec: Dict[str, Any]) -> str:
    return json.dumps({"template_id": spec.get("template_id"), "constraints": spec.get("constraints", {})}, ensure_ascii=False, sort_keys=True)


def _cap_specs_by_template(specs: List[Dict[str, Any]], level: str, rng: random.Random) -> List[Dict[str, Any]]:
    by_tpl: Dict[str, List[Dict[str, Any]]] = {}
    for s in specs:
        by_tpl.setdefault(s["template_id"], []).append(s)
    capped = []
    cap = int(DISH_MAX_TEMPLATE_CANDIDATES.get(level, 400))
    for tpl, arr in by_tpl.items():
        rng.shuffle(arr)
        capped.extend(arr[:cap])
    return capped


def _round_robin_specs(specs: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    by_tpl: Dict[str, List[Dict[str, Any]]] = {}
    for s in specs:
        by_tpl.setdefault(s["template_id"], []).append(s)
    for arr in by_tpl.values():
        rng.shuffle(arr)
    out = []
    keys = list(by_tpl)
    rng.shuffle(keys)
    while any(by_tpl.values()):
        for k in list(keys):
            if by_tpl.get(k):
                out.append(by_tpl[k].pop())
    return out


def _dedupe_specs(specs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out, seen = [], set()
    for s in specs:
        sig = _candidate_signature(s)
        if sig in seen:
            continue
        seen.add(sig)
        out.append(s)
    return out


def build_dishes_candidate_queue(level: str, rng: random.Random) -> List[Dict[str, Any]]:
    countries = _all_df(country_pool())
    cuisines = _all_df(cuisine_pool())
    ingredients = _all_df(ingredient_pool())
    types = _all_df(dish_type_pool())
    languages = _all_df(language_pool())
    seeds = [dict(x) for x in seed_dish_pool().to_dict("records")]
    continents = list(ANCHOR_CONTINENTS)
    orgs = list(ANCHOR_ORGS)

    specs: List[Dict[str, Any]] = []

    if level == "L1":
        for c in countries:
            for ing in ingredients:
                specs.append(_make_l1_country_ingredient(c, ing))
            for typ in types:
                specs.append(_make_l1_country_type(c, typ))
        for cuis in cuisines:
            for ing in ingredients:
                specs.append(_make_l1_cuisine_ingredient(cuis, ing))
            for typ in types:
                specs.append(_make_l1_cuisine_type(cuis, typ))
        for ing in ingredients:
            for typ in types:
                specs.append(_make_l1_ingredient_type(ing, typ))
        for cont in continents:
            for ing in ingredients:
                specs.append(_make_l1_continent_ingredient(cont, ing))

    elif level == "L2":
        for c in countries:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l2_country_ingredient_type(c, ing, typ))
        for cuis in cuisines:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l2_cuisine_ingredient_type(cuis, ing, typ))
        for cont in continents:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l2_continent_ingredient_type(cont, ing, typ))
        for org in orgs:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l2_org_ingredient_type(org, ing, typ))
        for lang in languages:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l2_language_ingredient_type(lang, ing, typ))

    elif level == "L3":
        for cont in continents:
            for cuis in cuisines:
                for ing in ingredients:
                    specs.append(_make_l3_continent_cuisine_ingredient(cont, cuis, ing))
        for org in orgs:
            for cuis in cuisines:
                for ing in ingredients:
                    specs.append(_make_l3_org_cuisine_ingredient(org, cuis, ing))
        for anchor in countries:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l3_border_ingredient_type(anchor, ing, typ))
        for seed in seeds:
            for typ in types:
                specs.append(_make_l3_seed_shared_ingredient_type(seed, typ))
        for lang in languages:
            for cuis in cuisines:
                for typ in types:
                    specs.append(_make_l3_language_cuisine_type(lang, cuis, typ))

    elif level == "L4":
        for anchor in countries:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l4_same_continent_ingredient_type(anchor, ing, typ))
        for anchor in countries:
            for cuis in cuisines:
                for ing in ingredients:
                    specs.append(_make_l4_shared_language_cuisine_ingredient(anchor, cuis, ing))
        for source in countries:
            for org in orgs:
                for typ in types:
                    specs.append(_make_l4_source_country_ingredient_org_type(source, org, typ))
        for seed in seeds:
            for org in orgs:
                for ing in ingredients:
                    specs.append(_make_l4_seed_type_org_ingredient(seed, org, ing))

    elif level == "L5":
        for anchor in countries:
            for org in orgs:
                for ing in ingredients:
                    for typ in types:
                        specs.append(_make_l5_same_continent_org_ingredient_type(anchor, org, ing, typ))
        for anchor in countries:
            for seed in seeds:
                for typ in types:
                    specs.append(_make_l5_shared_language_seed_ingredient_type(anchor, seed, typ))
        for border_anchor in countries:
            for source in countries:
                if source["qid"] == border_anchor["qid"]:
                    continue
                for typ in types:
                    specs.append(_make_l5_border_source_ingredient_type(border_anchor, source, typ))
        for org in orgs:
            for ing in ingredients:
                for typ in types:
                    specs.append(_make_l5_cuisine_country_match_org_ingredient_type(org, ing, typ))
        for seed in seeds:
            for org in orgs:
                for ing in ingredients:
                    specs.append(_make_l5_seed_type_org_shared_ingredient(seed, org, ing))

    specs = _dedupe_specs(specs)
    specs = _cap_specs_by_template(specs, level, rng)
    specs = _round_robin_specs(specs, rng)
    return specs[: int(DISH_MAX_CANDIDATES_PER_LEVEL.get(level, len(specs)))]


def _build_dish_record(idx: int, complexity: str, spec: Dict[str, Any], max_gold: int = DISH_GOLD_LOCAL_LIMIT) -> Optional[BenchmarkExample]:
    where_lines = spec["where_lines"]
    sparql = _select_gold_sparql(where_lines, limit=DISH_GOLD_QUERY_LIMIT)
    rows = rows_from_select(wd.sparql_select(sparql))
    qids, labels_ru, labels_en, stats = _gold_from_rows(rows, max_gold=max_gold)

    requested = int(spec["requested_count"])
    if len(qids) < max(requested, DISH_MIN_GOLD_BY_LEVEL.get(complexity, requested)):
        return None

    constraints_clean = _clean_constraints(spec["constraints"])
    template_id = spec["template_id"]
    template_family = spec["template_family"]
    rows_returned = len(rows)
    gold_truncated = rows_returned >= DISH_GOLD_QUERY_LIMIT or len(qids) >= max_gold

    meta = _gold_meta(
        template_id=template_id,
        template_family=template_family,
        constraints=constraints_clean,
        rows_returned=rows_returned,
        gold_total=len(qids),
        stats=stats,
        query_limit=DISH_GOLD_QUERY_LIMIT,
        local_limit=max_gold,
    )
    meta["patch_version"] = DISHES_PATCH_VERSION
    meta["candidate_constraints_signature"] = _candidate_signature(spec)

    return BenchmarkExample(
        id=f"dishes_{complexity.lower()}_{idx:04d}",
        domain=DISH_DOMAIN_NAME,
        complexity=complexity,
        query_text_ru=spec["query_text_ru"],
        query_text_en=spec["query_text_en"],
        constraints=constraints_clean,
        requested_count=requested,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        gold_answer_labels_en=labels_en,
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=complexity in {"L3", "L4", "L5"},
        template_id=template_id,
        template_family=template_family,
        gold_truncated=gold_truncated,
        ask_validator_sparql=_ask_validator_sparql(where_lines),
        local_validator=_local_validator_meta(constraints_clean),
        gold_collection_meta=meta,
        gold_answer_imdb_ids=[],
        gold_answer_imdb_titles=[],
    )


def _gold_overlap_reason(candidate_qids: Sequence[str], existing_records: Sequence[Dict[str, Any]]) -> Optional[str]:
    a = set(candidate_qids or [])
    if not a:
        return None
    for r in existing_records:
        b = set(r.get("gold_answer_qids") or [])
        if not b:
            continue
        inter = len(a & b)
        union = len(a | b)
        jacc = inter / union if union else 0.0
        containment = inter / min(len(a), len(b)) if min(len(a), len(b)) else 0.0
        size_ratio = min(len(a), len(b)) / max(len(a), len(b)) if max(len(a), len(b)) else 0.0
        if jacc >= DISH_DEDUP_GOLD_JACCARD_THRESHOLD:
            return f"gold_jaccard={jacc:.2f} with {r.get('id')}"
        if containment >= DISH_DEDUP_GOLD_CONTAINMENT_THRESHOLD and size_ratio >= DISH_DEDUP_GOLD_SIZE_RATIO_THRESHOLD:
            return f"gold_containment={containment:.2f}, size_ratio={size_ratio:.2f} with {r.get('id')}"
    return None


def generate_dishes_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 200) -> BenchmarkExample:
    # Backward-compatible single-example generator. Uses the queue and tries bounded candidates once each.
    queue = build_dishes_candidate_queue(complexity, rng)[:max_attempts]
    errors = []
    for spec in queue:
        try:
            ex = _build_dish_record(idx=idx, complexity=complexity, spec=spec)
            if ex is not None and len(ex.gold_answer_qids) >= ex.requested_count:
                return ex
        except KeyboardInterrupt:
            raise
        except Exception as e:
            errors.append(f"{type(e).__name__}: {str(e)[:180]}")
            continue
    raise RuntimeError(f"Failed to generate dishes example {complexity} after {len(queue)} queued candidates; recent errors: {errors[-5:]}")


if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["dishes"] = generate_dishes_example
    DOMAIN_GENERATORS["13_dishes"] = generate_dishes_example

print("✅ dishes v3 queue-based generator loaded")


✅ dishes v3 queue-based generator loaded


In [8]:
# Optional smoke test.
RUN_DISHES_SMOKE_TEST = False

if RUN_DISHES_SMOKE_TEST:
    test_rng = random.Random(42)
    ex = generate_dishes_example("L3", 1, test_rng, max_attempts=80)
    print(json.dumps(asdict(ex), ensure_ascii=False, indent=2)[:5000])


In [9]:

# ============================================================
# v4 hotfix: make L2 actually progress + improve candidate quality
# ============================================================
# Why this exists:
# v3 builds a queue, but the first L2 candidate can still be a very slow WDQS
# query (especially language/org + ingredient + dish_type). With helper retries,
# one bad candidate can look like an infinite hang. v4 does three things:
#   1) hard per-candidate timeout;
#   2) priority L2 candidates/templates that are faster and high-recall;
#   3) extra L2 ingredient-pair templates so L2 remains 3-criterion but not
#      always forced through another expensive P31/P279* dish-type path.

DISHES_PATCH_VERSION = "v4_l2_fast_quality"

# Keep L2 bounded. Slow candidates are skipped; generation should keep moving.
DISH_HARD_QUERY_TIMEOUT_SECONDS_BY_LEVEL = {"L1": 30, "L2": 28, "L3": 35, "L4": 40, "L5": 45}
DISH_MAX_CANDIDATES_PER_LEVEL.update({"L2": 1200})
DISH_MAX_TEMPLATE_CANDIDATES.update({"L2": 260})

# Network helper retries must not multiply a single slow candidate into many minutes.
try:
    wd.timeout = min(max(int(getattr(wd, "timeout", 20)), 15), 25)
    wd.max_retries = 1
except Exception:
    pass

import contextlib
import threading
try:
    import signal
except Exception:
    signal = None

class DishCandidateTimeout(TimeoutError):
    pass

@contextlib.contextmanager
def _dish_candidate_time_limit(seconds: int):
    """Unix/macOS hard timeout for a single WDQS candidate; no-op off main thread."""
    if not seconds or seconds <= 0 or signal is None or threading.current_thread() is not threading.main_thread():
        yield
        return
    old_handler = signal.getsignal(signal.SIGALRM)
    def _handler(signum, frame):
        raise DishCandidateTimeout(f"WDQS candidate exceeded {seconds}s")
    try:
        signal.signal(signal.SIGALRM, _handler)
        signal.setitimer(signal.ITIMER_REAL, float(seconds))
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_handler)


def _find_entity_by_en(items: List[Dict[str, str]], name: str) -> Optional[Dict[str, str]]:
    name_l = str(name).lower()
    for x in items:
        if str(x.get("en", "")).lower() == name_l:
            return x
    return None


def _make_l2_country_two_ingredients(c: Dict[str, str], ing1: Dict[str, str], ing2: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [
        _dish_base(),
        f"?item wdt:{P_COUNTRY_OF_ORIGIN} {_wd(c['qid'])} .",
        _ingredient_path("item", _wd(ing1["qid"])),
        _ingredient_path("item", _wd(ing2["qid"])),
    ]
    return _spec(
        "dishes_l2_country_two_ingredients", "origin_country_two_ingredients", where,
        {"kind": "dish", "country_of_origin": _en(c), "ingredient_1": _en(ing1), "ingredient_2": _en(ing2)},
        f"{_head_ru(n)}, происходящих из страны «{_ru(c)}» и содержащих ингредиенты «{_ru(ing1)}» и «{_ru(ing2)}»",
        f"{_head_en(n)} whose country of origin is {_en(c)} and that contain both {_en(ing1)} and {_en(ing2)}",
        n,
    )


def _make_l2_continent_two_ingredients(cont: Dict[str, str], ing1: Dict[str, str], ing2: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [
        _dish_base(),
        *_origin_country_lines("country"),
        f"?country wdt:{P_CONTINENT} {_wd(cont['qid'])} .",
        _ingredient_path("item", _wd(ing1["qid"])),
        _ingredient_path("item", _wd(ing2["qid"])),
    ]
    return _spec(
        "dishes_l2_continent_two_ingredients", "continent_two_ingredients", where,
        {"kind": "dish", "country_continent": _en(cont), "ingredient_1": _en(ing1), "ingredient_2": _en(ing2)},
        f"{_head_ru(n)}, происходящих из стран континента «{_ru(cont)}» и содержащих ингредиенты «{_ru(ing1)}» и «{_ru(ing2)}»",
        f"{_head_en(n)} whose country of origin is in {_en(cont)} and that contain both {_en(ing1)} and {_en(ing2)}",
        n,
    )


def _make_l2_cuisine_two_ingredients(cuis: Dict[str, str], ing1: Dict[str, str], ing2: Dict[str, str]) -> Dict[str, Any]:
    n = _requested_count("L2")
    where = [
        _dish_base(),
        f"?item wdt:{P_CUISINE} {_wd(cuis['qid'])} .",
        _ingredient_path("item", _wd(ing1["qid"])),
        _ingredient_path("item", _wd(ing2["qid"])),
    ]
    return _spec(
        "dishes_l2_cuisine_two_ingredients", "cuisine_two_ingredients", where,
        {"kind": "dish", "cuisine": _en(cuis), "ingredient_1": _en(ing1), "ingredient_2": _en(ing2)},
        f"{_head_ru(n)} кухни «{_ru(cuis)}», содержащих ингредиенты «{_ru(ing1)}» и «{_ru(ing2)}»",
        f"{_head_en(n)} from {_en(cuis)} cuisine that contain both {_en(ing1)} and {_en(ing2)}",
        n,
    )


# Save v3 implementations so we can reuse them for L1/L3/L4/L5 and most L2 specs.
_build_dishes_candidate_queue_v3 = build_dishes_candidate_queue


def _l2_priority_specs(rng: random.Random) -> List[Dict[str, Any]]:
    countries = _all_df(country_pool())
    cuisines = _all_df(cuisine_pool())
    ingredients = _all_df(ingredient_pool())
    types = _all_df(dish_type_pool())
    continents = list(ANCHOR_CONTINENTS)

    def C(name): return _find_entity_by_en(countries, name)
    def CU(name): return _find_entity_by_en(cuisines, name)
    def I(name): return _find_entity_by_en(ingredients, name)
    def T(name): return _find_entity_by_en(types, name)
    def K(name): return _find_entity_by_en(continents, name)

    specs: List[Dict[str, Any]] = []

    # Known high-recall, fairly fast L2 triples. If a few fail, timeout/skip keeps moving.
    triples_type = [
        ("continent", "Asia", "sugar", "dessert"),
        ("continent", "Asia", "flour", "bread"),
        ("continent", "Asia", "beef", "stew"),
        ("continent", "Europe", "cheese", "sandwich"),
        ("continent", "Europe", "butter", "dessert"),
        ("continent", "Africa", "flour", "bread"),
        ("country", "Spain", "flour", "bread"),
        ("country", "Japan", "fish", "soup"),
        ("country", "Indonesia", "sugar", "dessert"),
        ("country", "Nigeria", "fish", "soup"),
        ("country", "France", "butter", "dessert"),
        ("country", "Italy", "cheese", "pizza"),
        ("cuisine", "Japanese cuisine", "fish", "soup"),
        ("cuisine", "Spanish cuisine", "flour", "bread"),
        ("cuisine", "Mexican cuisine", "maize", "sandwich"),
        ("cuisine", "Indian cuisine", "rice", "dessert"),
    ]
    for kind, a, ing, typ in triples_type:
        ing_e, typ_e = I(ing), T(typ)
        if not ing_e or not typ_e:
            continue
        if kind == "continent":
            ent = K(a)
            if ent: specs.append(_make_l2_continent_ingredient_type(ent, ing_e, typ_e))
        elif kind == "country":
            ent = C(a)
            if ent: specs.append(_make_l2_country_ingredient_type(ent, ing_e, typ_e))
        elif kind == "cuisine":
            ent = CU(a)
            if ent: specs.append(_make_l2_cuisine_ingredient_type(ent, ing_e, typ_e))

    pairs = [
        ("continent", "Asia", "rice", "sugar"),
        ("continent", "Asia", "flour", "sugar"),
        ("continent", "Europe", "butter", "egg"),
        ("continent", "Europe", "cheese", "potato"),
        ("continent", "Africa", "flour", "sugar"),
        ("country", "Indonesia", "rice", "sugar"),
        ("country", "Spain", "flour", "salt"),
        ("country", "United States", "butter", "sugar"),
        ("country", "Japan", "rice", "fish"),
        ("country", "Italy", "cheese", "tomato"),
        ("cuisine", "Italian cuisine", "cheese", "tomato"),
        ("cuisine", "Indian cuisine", "rice", "sugar"),
        ("cuisine", "Japanese cuisine", "rice", "fish"),
        ("cuisine", "Turkish cuisine", "flour", "sugar"),
    ]
    for kind, a, ing1, ing2 in pairs:
        ing1_e, ing2_e = I(ing1), I(ing2)
        if not ing1_e or not ing2_e or ing1_e["qid"] == ing2_e["qid"]:
            continue
        if kind == "continent":
            ent = K(a)
            if ent: specs.append(_make_l2_continent_two_ingredients(ent, ing1_e, ing2_e))
        elif kind == "country":
            ent = C(a)
            if ent: specs.append(_make_l2_country_two_ingredients(ent, ing1_e, ing2_e))
        elif kind == "cuisine":
            ent = CU(a)
            if ent: specs.append(_make_l2_cuisine_two_ingredients(ent, ing1_e, ing2_e))

    specs = _dedupe_specs(specs)
    rng.shuffle(specs)
    return specs


def _append_l2_pair_specs(base: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    countries = _all_df(country_pool())
    cuisines = _all_df(cuisine_pool())
    ingredients = _all_df(ingredient_pool())
    continents = list(ANCHOR_CONTINENTS)

    # Avoid very generic pairs dominating every L2 while still giving enough recall.
    pair_specs: List[Dict[str, Any]] = []
    ing_pairs = []
    for i, ing1 in enumerate(ingredients):
        for ing2 in ingredients[i+1:]:
            if {ing1["en"], ing2["en"]} in [
                {"salt", "sugar"}, {"salt", "water"}, {"sugar", "water"}, {"flour", "salt"},
            ]:
                continue
            ing_pairs.append((ing1, ing2))
    rng.shuffle(ing_pairs)
    ing_pairs = ing_pairs[:80]

    for cont in continents:
        for ing1, ing2 in ing_pairs[:45]:
            pair_specs.append(_make_l2_continent_two_ingredients(cont, ing1, ing2))
    for c in countries:
        for ing1, ing2 in ing_pairs[:35]:
            pair_specs.append(_make_l2_country_two_ingredients(c, ing1, ing2))
    for cuis in cuisines:
        for ing1, ing2 in ing_pairs[:35]:
            pair_specs.append(_make_l2_cuisine_two_ingredients(cuis, ing1, ing2))

    pair_specs = _dedupe_specs(pair_specs)
    pair_specs = _cap_specs_by_template(pair_specs, "L2", rng)
    pair_specs = _round_robin_specs(pair_specs, rng)
    return base + pair_specs


def build_dishes_candidate_queue(level: str, rng: random.Random) -> List[Dict[str, Any]]:
    queue = _build_dishes_candidate_queue_v3(level, rng)
    if level != "L2":
        return queue

    priority = _l2_priority_specs(rng)
    queue = _append_l2_pair_specs(queue, rng)

    # Fast/simple L2 templates first; org/language L2 are allowed but delayed.
    rank = {
        "dishes_l2_continent_two_ingredients": 0,
        "dishes_l2_country_two_ingredients": 1,
        "dishes_l2_cuisine_two_ingredients": 2,
        "dishes_l2_continent_ingredient_type": 3,
        "dishes_l2_country_ingredient_type": 4,
        "dishes_l2_cuisine_ingredient_type": 5,
        "dishes_l2_language_ingredient_type": 8,
        "dishes_l2_org_ingredient_type": 9,
    }

    # Preserve randomness inside ranks, but avoid starting with the slowest query families.
    for pos, s in enumerate(queue):
        s["_v4_queue_pos"] = pos
    queue = sorted(queue, key=lambda s: (rank.get(s.get("template_id"), 6), s.get("_v4_queue_pos", 0)))
    for s in queue:
        s.pop("_v4_queue_pos", None)

    merged = _dedupe_specs(priority + queue)
    return merged[: int(DISH_MAX_CANDIDATES_PER_LEVEL.get("L2", len(merged)))]


def _build_dish_record(idx: int, complexity: str, spec: Dict[str, Any], max_gold: int = DISH_GOLD_LOCAL_LIMIT) -> Optional[BenchmarkExample]:
    where_lines = spec["where_lines"]
    sparql = _select_gold_sparql(where_lines, limit=DISH_GOLD_QUERY_LIMIT)
    seconds = int(DISH_HARD_QUERY_TIMEOUT_SECONDS_BY_LEVEL.get(complexity, 35))
    with _dish_candidate_time_limit(seconds):
        rows = rows_from_select(wd.sparql_select(sparql))
    qids, labels_ru, labels_en, stats = _gold_from_rows(rows, max_gold=max_gold)

    requested = int(spec["requested_count"])
    if len(qids) < max(requested, DISH_MIN_GOLD_BY_LEVEL.get(complexity, requested)):
        return None

    constraints_clean = _clean_constraints(spec["constraints"])
    template_id = spec["template_id"]
    template_family = spec["template_family"]
    rows_returned = len(rows)
    gold_truncated = rows_returned >= DISH_GOLD_QUERY_LIMIT or len(qids) >= max_gold

    meta = _gold_meta(
        template_id=template_id,
        template_family=template_family,
        constraints=constraints_clean,
        rows_returned=rows_returned,
        gold_total=len(qids),
        stats=stats,
        query_limit=DISH_GOLD_QUERY_LIMIT,
        local_limit=max_gold,
    )
    meta["patch_version"] = DISHES_PATCH_VERSION
    meta["candidate_constraints_signature"] = _candidate_signature(spec)
    meta["hard_timeout_seconds"] = seconds

    return BenchmarkExample(
        id=f"dishes_{complexity.lower()}_{idx:04d}",
        domain=DISH_DOMAIN_NAME,
        complexity=complexity,
        query_text_ru=spec["query_text_ru"],
        query_text_en=spec["query_text_en"],
        constraints=constraints_clean,
        requested_count=requested,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        gold_answer_labels_en=labels_en,
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=complexity in {"L3", "L4", "L5"},
        template_id=template_id,
        template_family=template_family,
        gold_truncated=gold_truncated,
        ask_validator_sparql=_ask_validator_sparql(where_lines),
        local_validator=_local_validator_meta(constraints_clean),
        gold_collection_meta=meta,
        gold_answer_imdb_ids=[],
        gold_answer_imdb_titles=[],
    )

print("✅ dishes v4 L2 fast-quality hotfix loaded")
print("patch:", DISHES_PATCH_VERSION)
print("WDQS timeout:", getattr(wd, "timeout", None), "max_retries:", getattr(wd, "max_retries", None))


✅ dishes v4 L2 fast-quality hotfix loaded
patch: v4_l2_fast_quality
WDQS timeout: 25 max_retries: 1


In [10]:

# ============================================================
# v3 incremental queue generation cell
# ============================================================
# This cell appends each accepted record immediately. If interrupted, rerun it: it resumes from JSONL.

OUT_DIR_DOMAIN = Path("out_wikidata_benchmark/domain_outputs")
OUT_DIR_DOMAIN.mkdir(parents=True, exist_ok=True)

DISHES_OUTPUT_PATH = OUT_DIR_DOMAIN / "dishes.jsonl"
DISHES_AUDIT_PATH = OUT_DIR_DOMAIN / "dishes_generation_audit.json"
DISHES_CHECKPOINT_PATH = OUT_DIR_DOMAIN / "dishes_generation_checkpoint.json"

# Backward-compatible aliases used by the report cell.
out_path = DISHES_OUTPUT_PATH
audit_path = DISHES_AUDIT_PATH
checkpoint_path = DISHES_CHECKPOINT_PATH

GENERATION_SEED = SEED
rng = random.Random(GENERATION_SEED)

print("output:", out_path.resolve())
print("audit:", audit_path.resolve())
print("checkpoint:", checkpoint_path.resolve())
print("patch:", DISHES_PATCH_VERSION)


def read_existing_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    return records


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()


def write_json(path: Path, obj: Dict[str, Any]) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()


def make_record_key(r: Dict[str, Any]) -> Tuple[str, str, str]:
    return (
        str(r.get("complexity") or ""),
        str(r.get("query_text_ru") or ""),
        json.dumps(r.get("constraints", {}) or {}, ensure_ascii=False, sort_keys=True),
    )


def _level_numeric_id(r: Dict[str, Any], level: str) -> int:
    m = re.search(rf"dishes_{level.lower()}_(\d+)$", str(r.get("id", "")))
    return int(m.group(1)) if m else 0


def current_audit(records: List[Dict[str, Any]], skipped: List[Dict[str, Any]], queue_meta: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "domain": DISH_DOMAIN_NAME,
        "patch_version": DISHES_PATCH_VERSION,
        "target_plan": TARGET_PLAN_DISHES,
        "records_total": len(records),
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in records)),
        "template_counts": dict(Counter(r.get("template_id") for r in records)),
        "template_family_counts": dict(Counter(r.get("template_family") for r in records)),
        "gold_truncated_count": sum(1 for r in records if r.get("gold_truncated")),
        "avg_gold_count_by_complexity": {
            level: round(sum(len(r.get("gold_answer_qids", [])) for r in records if r.get("complexity") == level) / max(1, sum(1 for r in records if r.get("complexity") == level)), 2)
            for level in TARGET_PLAN_DISHES
        },
        "queue_meta": queue_meta,
        "skipped_count": len(skipped),
        "skipped_preview": skipped[-160:],
        "output_path": str(out_path),
        "audit_path": str(audit_path),
        "checkpoint_path": str(checkpoint_path),
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }


def validate_record_schema(r: Dict[str, Any]) -> None:
    dummy = BenchmarkExample(
        id="dummy", domain="dummy", complexity="L1", query_text_ru="", constraints={}, requested_count=1,
        gold_answer_qids=[], gold_answer_labels_ru=[], sparql_query="", created_at=utc_now_z(),
    )
    expected = list(asdict(dummy).keys())
    got = list(r.keys())
    if set(got) != set(expected):
        raise ValueError(f"Schema mismatch. Missing={sorted(set(expected)-set(got))}; extra={sorted(set(got)-set(expected))}")
    if not r.get("query_text_en"):
        raise ValueError("empty query_text_en")
    if not isinstance(r.get("constraints"), dict):
        raise ValueError("constraints must be dict")
    bad_constraint_keys = [k for k in r["constraints"] if k.endswith("_qid") or k.endswith("_qids") or k.endswith("_ru") or k.endswith("_en") or k in {"sparql_query", "where_lines"}]
    if bad_constraint_keys:
        raise ValueError(f"dirty constraint keys: {bad_constraint_keys}")
    if len(r.get("gold_answer_qids", [])) < int(r.get("requested_count", 0)):
        raise ValueError("not enough gold answers")
    if len(r.get("gold_answer_qids", [])) != len(r.get("gold_answer_labels_ru", [])) or len(r.get("gold_answer_qids", [])) != len(r.get("gold_answer_labels_en", [])):
        raise ValueError("gold qids/labels length mismatch")
    if len(r.get("gold_answer_qids", [])) != len(set(r.get("gold_answer_qids", []))):
        raise ValueError("duplicate gold qids")


records = read_existing_jsonl(out_path)
seen_keys = {make_record_key(r) for r in records}
skipped: List[Dict[str, Any]] = []
existing_counts = Counter(r.get("complexity") for r in records)
queue_meta: Dict[str, Any] = {}

print("existing records:", len(records))
print("existing counts:", dict(existing_counts))

for i, rec in enumerate(records[:]):
    try:
        validate_record_schema(rec)
    except Exception as e:
        skipped.append({"complexity": rec.get("complexity"), "reason": "existing_schema_warning", "row_index": i, "error": str(e)[:500]})

overall_total = sum(TARGET_PLAN_DISHES.values())
already_done_total = sum(min(existing_counts.get(level, 0), target) for level, target in TARGET_PLAN_DISHES.items())
overall_bar = tqdm(total=overall_total, initial=already_done_total, desc="dishes total", position=0)

try:
    for complexity, target_n in TARGET_PLAN_DISHES.items():
        already_done = existing_counts.get(complexity, 0)
        if already_done >= target_n:
            print(f"SKIP: dishes:{complexity} already has {already_done}/{target_n}")
            continue

        level_next_idx = max([_level_numeric_id(r, complexity) for r in records] + [0]) + 1
        level_records_count = already_done
        accepted_this_run = 0

        queue = build_dishes_candidate_queue(complexity, rng)
        queue_meta[complexity] = {"candidates": len(queue), "accepted_this_run": 0, "tried": 0}
        print(f"dishes:{complexity} candidate queue:", len(queue), "target need:", target_n - already_done)

        level_bar = tqdm(total=target_n, initial=already_done, desc=f"dishes:{complexity}", position=1, leave=True)

        for tried, spec in enumerate(queue, start=1):
            if level_records_count >= target_n:
                break
            queue_meta[complexity]["tried"] = tried
            level_bar.set_postfix({"ok": level_records_count, "target": target_n, "tried": tried, "candidates": len(queue)})

            key0 = (complexity, "", json.dumps(spec.get("constraints", {}) or {}, ensure_ascii=False, sort_keys=True))
            if any(k[0] == complexity and k[2] == key0[2] for k in seen_keys):
                skipped.append({"complexity": complexity, "reason": "duplicate_constraints_pre_wdqs", "template_id": spec.get("template_id"), "constraints": spec.get("constraints")})
                continue

            try:
                ex = _build_dish_record(idx=level_next_idx, complexity=complexity, spec=spec)
                if ex is None:
                    skipped.append({"complexity": complexity, "reason": "not_enough_gold", "template_id": spec.get("template_id"), "constraints": spec.get("constraints")})
                    continue

                r = _example_to_dict(ex)
                validate_record_schema(r)
                key = make_record_key(r)
                if key in seen_keys:
                    skipped.append({"complexity": complexity, "reason": "duplicate", "query_text_ru": r.get("query_text_ru"), "constraints": r.get("constraints")})
                    continue

                overlap = _gold_overlap_reason(r.get("gold_answer_qids", []), [x for x in records if x.get("complexity") == complexity])
                if overlap:
                    skipped.append({"complexity": complexity, "reason": "gold_overlap", "detail": overlap, "template_id": r.get("template_id"), "constraints": r.get("constraints")})
                    continue

                seen_keys.add(key)
                records.append(r)
                append_jsonl(out_path, r)
                level_next_idx += 1
                level_records_count += 1
                accepted_this_run += 1
                queue_meta[complexity]["accepted_this_run"] = accepted_this_run
                level_bar.update(1)
                overall_bar.update(1)

                audit = current_audit(records, skipped, queue_meta)
                write_json(audit_path, audit)
                write_json(checkpoint_path, {"last_record": r, "audit": audit})
                level_bar.set_postfix({"ok": level_records_count, "target": target_n, "tried": tried, "tpl": r.get("template_id"), "gold": len(r.get("gold_answer_qids", []))})

            except KeyboardInterrupt:
                raise
            except Exception as e:
                skipped.append({"complexity": complexity, "reason": type(e).__name__, "template_id": spec.get("template_id"), "constraints": spec.get("constraints"), "error": str(e)[:800]})
                audit = current_audit(records, skipped, queue_meta)
                write_json(audit_path, audit)
                continue

        level_bar.close()
        if level_records_count < target_n:
            print(f"WARN: only {level_records_count}/{target_n} for dishes:{complexity} after trying {queue_meta[complexity]['tried']}/{len(queue)} candidates")
        else:
            print(f"OK: dishes:{complexity} {level_records_count}/{target_n}; accepted_this_run={accepted_this_run}")

finally:
    overall_bar.close()
    audit = current_audit(records, skipped, queue_meta)
    write_json(audit_path, audit)
    print()
    print("saved incrementally:", out_path.resolve())
    print("audit:", audit_path.resolve())
    print("checkpoint:", checkpoint_path.resolve())
    print("records:", len(records))
    print("counts:", dict(Counter(r.get("complexity") for r in records)))
    print("template counts:", dict(Counter(r.get("template_id") for r in records)))
    print("skipped:", len(skipped))


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes_generation_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes_generation_checkpoint.json
patch: v4_l2_fast_quality
existing records: 0
existing counts: {}


dishes total:   0%|          | 0/120 [00:00<?, ?it/s]

dishes:L1 candidate queue: 900 target need: 15


dishes:L1: 100%|██████████| 15/15 [00:00<00:00, 64.27it/s, ok=15, target=15, tried=118, tpl=dishes_l1_country_type, gold=23]


OK: dishes:L1 15/15; accepted_this_run=15
dishes:L2 candidate queue: 1200 target need: 20


dishes total:  18%|█▊        | 22/120 [21:43<1:36:45, 59.24s/it] 



saved incrementally: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes_generation_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/dishes_generation_checkpoint.json
records: 22
counts: {'L1': 15, 'L2': 7}
template counts: {'dishes_l1_continent_ingredient': 7, 'dishes_l1_country_ingredient': 2, 'dishes_l1_ingredient_type': 3, 'dishes_l1_country_type': 3, 'dishes_l2_continent_two_ingredients': 6, 'dishes_l2_continent_ingredient_type': 1}
skipped: 273


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Final JSONL schema / consistency report
# ============================================================
# Run after generation to inspect the output quickly.

REPORT_PATH = Path("out_wikidata_benchmark/domain_outputs/dishes.jsonl")

if REPORT_PATH.exists():
    rows = read_existing_jsonl(REPORT_PATH)
    print("rows:", len(rows))
    print("counts:", dict(Counter(r.get("complexity") for r in rows)))
    print("template counts:", dict(Counter(r.get("template_id") for r in rows)))
    print("gold truncated:", sum(1 for r in rows if r.get("gold_truncated")))

    schema_errors = []
    for i, r in enumerate(rows):
        try:
            validate_record_schema(r)
        except Exception as e:
            schema_errors.append((i, str(e)))
    print("schema errors:", len(schema_errors))
    if schema_errors[:10]:
        print(schema_errors[:10])
else:
    print("No dishes.jsonl yet. Run the generation cell first.")
